# Bước 1: Build BM25 Index (CHẠY TRÊN CPU - 30GB RAM)
Tắt GPU để có 30GB RAM. Load dữ liệu và Build Index, sau đó lưu ra Output và đẩy lên Dataset.


In [ ]:
!pip install -q bm25s pandas pyarrow


In [ ]:
import os
# Giảm luồng để chống tràn RAM khi chạy đa luồng
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import pandas as pd
import glob
import bm25s
import gc

corpus_files = glob.glob("/kaggle/input/r2ai-stage-3-raw-corpus-text/**/*.parquet", recursive=True)
if not corpus_files: print("Không tìm thấy file parquet!")

doc_ids = []
corpus_texts = []
print("1. Đang nạp bài báo vào RAM...")
for f in corpus_files:
    df = pd.read_parquet(f, columns=["id", "text"]).dropna(subset=["text"])
    doc_ids.extend(df["id"].astype(str).tolist())
    corpus_texts.extend(df["text"].astype(str).tolist())

print("2. Tokenize (Siêu tốc bằng C++)...")
corpus_tokens = bm25s.tokenize(corpus_texts)

print("3. Xóa văn bản gốc khỏi RAM để nhường chỗ cho BM25 Index...")
del corpus_texts
gc.collect()

print("4. Build BM25 Index...")
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

print("5. Lưu Index ra đĩa...")
retriever.save("bm25_index_dir")
print("Thành công! Hãy tải thư mục bm25_index_dir/ về và tạo Kaggle Dataset.")
